# Bước 2: Inference & Reranker Submission (CHẠY TRÊN GPU T4/P100)


In [ ]:
!pip install -q bm25s pandas pyarrow sentence-transformers accelerate torch


In [ ]:
import pandas as pd
import glob
import bm25s

print("1. Load toàn bộ văn bản vào RAM (Chỉ tốn ~3GB nên vừa vặn 13-16GB RAM của GPU Kaggle)...")
corpus_files = glob.glob("/kaggle/input/r2ai-stage-3-raw-corpus-text/**/*.parquet", recursive=True)
doc_ids = []
corpus_texts = []
for f in corpus_files:
    df = pd.read_parquet(f, columns=["id", "text"]).dropna(subset=["text"])
    doc_ids.extend(df["id"].astype(str).tolist())
    corpus_texts.extend(df["text"].astype(str).tolist())

print("2. Load BM25 Index từ Dataset (Dùng mmap=True nên ngốn 0GB RAM)...")
INDEX_DIR = "/kaggle/input/TEN-DATASET-CUA-BAN/bm25_index_dir" # Sửa lại tên thư mục Dataset của bạn!
retriever = bm25s.BM25.load(INDEX_DIR, mmap=True)


In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
    print("Đã nạp HF_TOKEN từ Kaggle Secrets.")
except Exception as e:
    print("Không tìm thấy Kaggle Secrets. Bỏ qua...")

from sentence_transformers import CrossEncoder
import torch
print("Đang tải mô hình Reranker (BGE-M3)...")
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=1024, model_kwargs={"torch_dtype": torch.float16})


In [ ]:
import json
import numpy as np

df_queries = pd.read_parquet("/kaggle/input/TÊN-DATASET-CUỘC-THI/query.parquet") # Sửa tên Dataset cuộc thi
results = []
print("4. Đang chạy tìm kiếm Inference...")

for idx, row in df_queries.iterrows():
    q_id = int(row["id"])
    q_text = str(row["query"])
    q_tokens = bm25s.tokenize([q_text])
    
    # Lấy top 100 bằng BM25
    doc_indices, scores = retriever.retrieve(q_tokens, k=100)
    top_k_indices = doc_indices[0]
    
    # Reranking
    top_100_texts = [corpus_texts[i] for i in top_k_indices]
    pairs = [[q_text, txt] for txt in top_100_texts]
    rerank_scores = reranker.predict(pairs, batch_size=4)
    best_local_indices = np.argsort(rerank_scores)[::-1]
    reranked_top_k_indices = [top_k_indices[i] for i in best_local_indices]
    
    # Lấy Top 5 nộp bài
    result_doc_ids = [int(doc_ids[i]) for i in reranked_top_k_indices[:5]]
    
    results.append({"id": q_id, "relevant_docs": result_doc_ids, "relevant_chunks": []})
    if (idx + 1) % 10 == 0: print(f"Đã xử lý {idx + 1}/{len(df_queries)} truy vấn...")

with open("submission.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
import zipfile
with zipfile.ZipFile("submission.zip", "w") as zf:
    zf.write("submission.json")
print("Đã lưu submission.zip thành công!")
